## Case studies for missing variants

Here we provide case studies for the different missing variant types.

In [ ]:
import oncophylo as op
import numpy as np
import pandas as pd
import anndata as ad
import os, sys
import matplotlib.pyplot as plt
from pathlib import Path
from anndata import io
import h5py, json

from utils import load_tree
from simulation_utils import mae, fer

In [ ]:
def run_LoPhy_with_missing_variant(adata, 
                                   case_study_folder, 
                                   case_study_name,
                                   variant_to_drop=None, 
                                   sample_to_drop_from=None, 
                                   amplicon_dropout=False):
        cells = adata.obs.index
        mutations = adata.var.index
        character_matrix = pd.DataFrame(adata.X.copy(), index=cells, columns=mutations)
        variant_reads_df = pd.DataFrame(adata.layers[op.ul.DATA.VARIANT_READS_CORRUPT].copy(), index=cells, columns=mutations)
        total_reads_df = pd.DataFrame(adata.layers[op.ul.DATA.TOTAL_READS_CORRUPT].copy(), index=cells, columns=mutations)
        regions_df = adata.uns[op.ul.DATA.REGION_READS].copy()
        regions_df.columns = regions_df.columns.astype(int)
        meta_df = adata.var.set_index("CHR").drop(columns="VARIANT_TYPE")
        cell_samples = adata.obs[op.ul.DATA.CELL_SAMPLE]
    
        true_genotypes = pd.DataFrame(adata.layers[op.ul.DATA.TRUE_DATA], index=cells, columns=mutations)
        B_true = pd.DataFrame(adata.uns[op.ul.DATA.MUTANT_COPY_NUMBERS][adata.obs[op.ul.DATA.CLUSTER_ID]], index=cells)
        A_true = pd.DataFrame(adata.uns[op.ul.DATA.TOTAL_COPY_NUMBERS][:,adata.obs[op.ul.DATA.CLUSTER_ID]].T, index=cells)
        
        character_matrix = character_matrix.replace(2,1).replace(3,-1)
        if variant_to_drop is not None and sample_to_drop_from is not None:
            cell_mask = (cell_samples == sample) & \
                        ((character_matrix[variant] == 1) | (character_matrix[variant] == 3) | (character_matrix[variant] == -1))
            character_matrix.loc[cell_mask, variant_to_drop] = 3
            variant_reads_df.loc[cell_mask, variant_to_drop] = 0
            if amplicon_dropout:
                total_reads_df.loc[cell_mask, variant_to_drop] //= 2
                region_name = "_".join([adata.var.loc[variant, "CHR"], adata.var.loc[variant, "REGION"]])
                regions_df.loc[region_name, cell_mask.values] -=  total_reads_df.loc[cell_mask, variant_to_drop]

        LoPhy_folder = os.path.join(case_study_folder, case_study_name)
        sol_LoPhy = op.tl.solver.LoPhy(character_matrix.replace(2,1).replace(3,-1), 
                                       variant_reads_df, 
                                       total_reads_df,                        
                                       regions_df, 
                                       meta_df,
                                       cell_samples=cell_samples,
                                       remove_temp_dir=True,
                                       destination_dir=LoPhy_folder,
                                       num_restarts=3,
                                       )

        # process LoPhy's results
        T_LoPhy = sol_LoPhy[op.ul.DATA.CELL_TREE]
        cell_assignments_LoPhy = np.array(T_LoPhy.graph["cell_assignments"], dtype=int)
        alt_cns = np.array(T_LoPhy.graph[op.ul.DATA.MUTANT_COPY_NUMBERS], dtype=int)
        total_cns = np.array(T_LoPhy.graph[op.ul.DATA.TOTAL_COPY_NUMBERS], dtype=int)
        B_pred_LoPhy = pd.DataFrame(alt_cns[cell_assignments_LoPhy], index=cells)
        A_pred_LoPhy = pd.DataFrame(total_cns[cell_assignments_LoPhy], index=cells)
        predicted_genotypes_LoPhy = sol_LoPhy[op.ul.DATA.PRED_DATA].astype(int)

        return [
                sol_LoPhy, 
                sol_LoPhy[op.ul.EVAL_KEYS.RUNTIME], 
                mae(B_true.values, B_pred_LoPhy.values), 
                mae(A_true.values, A_pred_LoPhy.values),
                fer(A_true=A_true.values, B_true=B_true.values, A_pred=A_pred_LoPhy.values, B_pred=B_pred_LoPhy.values, cell_samples=cell_samples),
               ]


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

def plot_clone_fractions_bar(
    cell_samples,
    cell_assignments,
    ax=None,
    colors=None,
    label_threshold=0.05,
    save_path=None
):
    """
    Plot stacked barplots of clone fractions at each sampling time.

    Parameters
    ----------
    cell_samples : array-like
        Sample/timepoint for each cell.
    cell_assignments : array-like
        Clone assignment for each cell.
    ax : matplotlib.axes.Axes, optional
    colors : dict or list, optional
        Either {clone_id: color} or list of colors.
    label_threshold : float, default=0.05
        Fractions >= this value are labeled inside the bar.
        Smaller fractions are listed above the bar.
    """

    if ax is None:
        fig, ax = plt.subplots(figsize=(9, 6))

    cell_samples = np.asarray(cell_samples)
    cell_assignments = np.asarray(cell_assignments)

    # Preserve order of appearance
    sample_order = list(dict.fromkeys(cell_samples))
    clones = np.sort(np.unique(cell_assignments))

    counts = np.zeros((len(clones), len(sample_order)), dtype=int)

    for j, sample in enumerate(sample_order):
        mask = cell_samples == sample
        for i, clone in enumerate(clones):
            counts[i, j] = np.sum(cell_assignments[mask] == clone)

    totals = counts.sum(axis=0)
    fractions = counts / totals

    bottom = np.zeros(len(sample_order))

    # Draw stacked bars
    for i, clone in enumerate(clones):

        if colors is None:
            color = None
        elif isinstance(colors, dict):
            color = colors[clone]
        else:
            color = colors[i]

        ax.bar(
            np.arange(len(sample_order)),
            fractions[i],
            bottom=bottom,
            label=f"Clone {clone}",
            color=color,
            edgecolor="white",
            linewidth=0.75,
        )

        # Label large populations inside the bars
        for j, frac in enumerate(fractions[i]):
            if frac >= label_threshold:
                ax.text(
                    j,
                    bottom[j] + frac / 2,
                    f"{100 * frac:.0f}%",
                    ha="center",
                    va="center",
                    fontsize=9,
                    color="black",
                )

        bottom += fractions[i]

    # Annotate sample sizes and rare clones
    for j, n in enumerate(totals):

        lines = [f"n={n}"]

        small = []
        for i, clone in enumerate(clones):
            frac = fractions[i, j]

            if 0 < frac < label_threshold:
                if frac < 0.01:
                    pct = "<1%"
                else:
                    pct = f"{100 * frac:.0f}%"

                small.append(f"Clone {clone}: {pct} ({counts[i,j]}/{n}) ")

        if small:
            lines.append("<5%:")
            lines.extend(small)

        ax.text(
            j,
            1.02,
            "\n".join(lines),
            ha="center",
            va="bottom",
            fontsize=8,
            clip_on=False,
        )

    ax.set_xticks(np.arange(len(sample_order)))
    ax.set_xticklabels(np.arange(1,len(sample_order)+1))

    ax.set_ylim(0, 1.22)
    ax.set_ylabel("Cells (%)")
    ax.set_xlabel("Sample")
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))

    ax.legend(
        title="Clone",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=False,
    )
    
    if save_path is not None:
        plt.savefig(save_path)
        
    return ax

In [ ]:
def prep_tree(T):
    
    import re

    def sort_node_labels(G):
        G = G.copy()

        for node in G.nodes:
            label = G.nodes[node].get("label", "")

            # Remove surrounding quotes if present
            if label.startswith('"') and label.endswith('"'):
                label = label[1:-1]

            entries = [e.strip() for e in label.split("\\n") if e.strip()]

            def sort_key(entry):
                # SNVx (chry)
                m = re.search(r"chr(\d+)", entry)
                if m:
                    return (0, int(m.group(1)))

                # Gain/Loss x-y
                m = re.search(r"(Gain|Loss)\s+(\d+)-", entry)
                if m:
                    return (1, int(m.group(2)), m.group(1))

                return (2, entry)

            entries.sort(key=sort_key)

            G.nodes[node]["label"] = '"' + "\n".join(entries) + '"'

        return G
    T = T.copy()
    T.nodes["Clone_0"]["label"] = "root"
    
    # update node label format
    for n in T.nodes:
        T.nodes[n]["label"] = T.nodes[n]["label"].replace("_p", " p.")
        
    T = sort_node_labels(T)
        
    return T

In [ ]:
import re

def format_label(label, already_formatted=False):
    """
    Formats node labels by converting SNVs to the form
    'SNVX (chrY)' and sorting them by chromosome.
    Works for both HTML (<br/>) and plain (\n) labels.
    """

    # Determine label format
    is_html = label.startswith("<") and label.endswith(">")
    is_quoted = label.startswith('"') and label.endswith('"')

    if is_html:
        label = label[1:-1]
        items = [x.strip() for x in label.split("<br/>") if x.strip()]
    elif is_quoted:
        label = label[1:-1]
        items = [x.strip() for x in label.split("\n") if x.strip()]
    else:
        items = [x.strip() for x in label.split("\n") if x.strip()]

    snvs = []
    cnas = []
    others = []

    for item in items:

        # Already in form "SNV4 (chr4)"
        m = re.match(r"SNV(\d+)\s*\(chr(\d+)\)", item)
        if m:
            snv = int(m.group(1))
            chrom = int(m.group(2))
            snvs.append((chrom, snv, f"SNV{snv} (chr{chrom})"))
            continue

        # Already in form "SNV4 chr4"
        m = re.match(r"SNV(\d+)\s+chr(\d+)", item)
        if m:
            snv = int(m.group(1))
            chrom = int(m.group(2))
            snvs.append((chrom, snv, f"SNV{snv} (chr{chrom})"))
            continue

        # Old form "4-SNV4"
        m = re.match(r"(\d+)-SNV(\d+)", item)
        if m:
            chrom = int(m.group(1))
            snv = int(m.group(2))
            snvs.append((chrom, snv, f"SNV{snv} (chr{chrom})"))
            continue
            
        m = re.match(r"(Gain|Loss)\s+(\d+)-(\d+)", item)
        if m:
            cna_type = m.group(1)
            chrom = int(m.group(2))
            allele = int(m.group(3))
            if allele == 0:
                allele = "REF"
            else:
                allele = "ALT"
            cnas.append((chrom, allele, f"{cna_type} {chrom} {allele}"))
            continue

        others.append(item)

    # Sort first by chromosome, then by SNV number
    snvs.sort(key=lambda x: (x[0], x[1]))
    cnas.sort(key=lambda x: (x[0], x[1]))


    lines = [x[2] for x in snvs] + [x[2] for x in cnas] + others

    if is_html:
        return "<" + "<br/>".join(lines) + ">"
    elif is_quoted:
        return '"' + "\n".join(lines) + '"'
    else:
        return "\n".join(lines)

In [ ]:
def relabel_nodes(T, already_formatted=False):
    T = T.copy()
    for node in T.nodes:
        if "label" in T.nodes[node]:
            T.nodes[node]["label"] = format_label(
                T.nodes[node]["label"], already_formatted
            )
    return T

In [ ]:
case_studies_directory = os.path.join(os.getcwd(), "missing_variants_case_studies")

In [ ]:
fig_directory = os.path.join(os.getcwd(), "..", "paper", "sim_results", "missing_variants_case_studies")

# Variant is missing when variant-harboring clone is near extinction 

## n9000_m20_CNA3_clones7_r20_s3_trial1

In [ ]:
dataset = "n9000_m20_CNA3_clones7_r20_s3_trial1"
dataset_path = os.path.join(case_studies_directory, dataset)
h5ad_fn = os.path.join(dataset_path, "adata.h5ad")
adata = io.read_h5ad(h5ad_fn)

In [ ]:
T_true = load_tree(adata, op.ul.DATA.MUTATION_TREE).copy()

In [ ]:
cell_samples = adata.obs["cell_sample"].values
cell_assignments = np.array(T_true.graph["cell_assignments"], dtype=int)

In [ ]:
mutant_copy_numbers = adata.uns['mutant_copy_numbers']

In [ ]:
mutant_copy_numbers_df = pd.DataFrame(mutant_copy_numbers, columns=adata.var_names, index=[f"Clone_{i}" for i in range(mutant_copy_numbers.shape[0])])

In [ ]:
plot_clone_fractions_bar(cell_samples, cell_assignments, save_path=os.path.join(fig_directory, dataset, "simulation_overview.svg"))

In [ ]:
op.pl.show_tree(T_true)

In [ ]:
variant = "SNV5"
sample = 1

In [ ]:
LoPhy_results = run_LoPhy_with_missing_variant(adata, 
                                         case_study_folder=dataset_path, 
                                         case_study_name=f"LoPhy_original_data")

In [ ]:
LoPhy_results_missing_var = run_LoPhy_with_missing_variant(adata, 
                                         case_study_folder=dataset_path, 
                                         case_study_name=f"LoPhy_{variant}_missing_sample{sample}",
                                         variant_to_drop=variant, 
                                         sample_to_drop_from=sample, 
                                         amplicon_dropout=False)

In [ ]:
print(f"Original data\n MCN-MAE: {LoPhy_results[2]} \nTCN-MAE: {LoPhy_results[3]}")
print(f"Missing variant data\n MCN-MAE: {LoPhy_results_missing_var[2]} \nTCN-MAE: {LoPhy_results_missing_var[3]}")

In [ ]:
_, T_original_data = op.io.load_dot(os.path.join(dataset_path, "LoPhy_original_data", "out_ml0.gv"), 
                                   _type="cell_tree")
_, T_missing_variant = op.io.load_dot(os.path.join(dataset_path, f"LoPhy_{variant}_missing_sample{sample}", "out_ml0.gv"), 
                                   _type="cell_tree")

In [ ]:
T_original_data = prep_tree(T_original_data)
T_missing_variant = prep_tree(T_missing_variant)

In [ ]:
T_true.nodes["0"]["label"] = "root"

In [ ]:
T_true_relabeled = relabel_nodes(T_true)

In [ ]:
for u, v in T_true_relabeled.edges():
    T_true_relabeled[u][v][0].pop("weight", None)
    T_true_relabeled[u][v][0].pop("penwidth", None)

In [ ]:
op.pl.show_tree(T_true_relabeled, save_path=os.path.join(fig_directory, dataset, "true_tree.svg"))

In [ ]:
op.pl.show_tree(T_original_data, save_path=os.path.join(fig_directory, dataset, "LoPhy_tree_original_data.svg"))

In [ ]:
op.pl.show_tree(T_missing_variant, save_path=os.path.join(fig_directory, dataset, "LoPhy_tree_missing_variant.svg"))

## n12000_m20_CNA3_clones8_r20_s4_trial0

In [ ]:
dataset = "n12000_m20_CNA3_clones8_r20_s4_trial0"
dataset_path = os.path.join(case_studies_directory, dataset)
h5ad_fn = os.path.join(dataset_path, "adata.h5ad")
adata = io.read_h5ad(h5ad_fn)

In [ ]:
T_true = load_tree(adata, op.ul.DATA.MUTATION_TREE).copy()

In [ ]:
cell_samples = adata.obs["cell_sample"].values
cell_assignments = np.array(T_true.graph["cell_assignments"], dtype=int)

In [ ]:
np.unique(cell_assignments[cell_samples==2], return_counts=True)

In [ ]:
mutant_copy_numbers = adata.uns['mutant_copy_numbers']

In [ ]:
mutant_copy_numbers_df = pd.DataFrame(mutant_copy_numbers, columns=adata.var_names, index=[f"Clone_{i}" for i in range(mutant_copy_numbers.shape[0])])

In [ ]:
plot_clone_fractions_bar(cell_samples, cell_assignments, save_path=os.path.join(fig_directory, dataset, "simulation_overview.svg"))

In [ ]:
op.pl.show_tree(T_true)

In [ ]:
mutant_copy_numbers_df

In [ ]:
variant = "SNV4"
sample = 2

In [ ]:
LoPhy_results = run_LoPhy_with_missing_variant(adata, 
                                         case_study_folder=dataset_path, 
                                         case_study_name=f"LoPhy_original_data")

In [ ]:
LoPhy_results_missing_var = run_LoPhy_with_missing_variant(adata, 
                                         case_study_folder=dataset_path, 
                                         case_study_name=f"LoPhy_{variant}_missing_sample{sample}",
                                         variant_to_drop=variant, 
                                         sample_to_drop_from=sample, 
                                         amplicon_dropout=False)

In [ ]:
print(f"Original data\n MCN-MAE: {LoPhy_results[2]} \nTCN-MAE: {LoPhy_results[3]}")
print(f"Missing variant data\n MCN-MAE: {LoPhy_results_missing_var[2]} \nTCN-MAE: {LoPhy_results_missing_var[3]}")

In [ ]:
_, T_original_data = op.io.load_dot(os.path.join(dataset_path, "LoPhy_original_data", "out_ml0.gv"), 
                                   _type="cell_tree")
_, T_missing_variant = op.io.load_dot(os.path.join(dataset_path, f"LoPhy_{variant}_missing_sample{sample}", "out_ml0.gv"), 
                                   _type="cell_tree")

In [ ]:
T_original_data = prep_tree(T_original_data)
T_missing_variant = prep_tree(T_missing_variant)

In [ ]:
T_true.nodes["0"]["label"] = "root"

In [ ]:
T_true_relabeled = relabel_nodes(T_true)

In [ ]:
for u, v in T_true_relabeled.edges():
    T_true_relabeled[u][v][0].pop("weight", None)
    T_true_relabeled[u][v][0].pop("penwidth", None)

In [ ]:
op.pl.show_tree(T_true_relabeled, save_path=os.path.join(fig_directory, dataset, "true_tree.svg"))

In [ ]:
op.pl.show_tree(T_original_data, save_path=os.path.join(fig_directory, dataset, "LoPhy_tree_original_data.svg"))

In [ ]:
op.pl.show_tree(T_missing_variant, save_path=os.path.join(fig_directory, dataset, "LoPhy_tree_missing_variant.svg"))

## n15000_m20_CNA3_clones9_r20_s5_trial9

In [ ]:
dataset = "n15000_m20_CNA3_clones9_r20_s5_trial9"
dataset_path = os.path.join(case_studies_directory, dataset)
h5ad_fn = os.path.join(dataset_path, "adata.h5ad")
adata = io.read_h5ad(h5ad_fn)

In [ ]:
T_true = load_tree(adata, op.ul.DATA.MUTATION_TREE).copy()

In [ ]:
cell_samples = adata.obs["cell_sample"].values
cell_assignments = np.array(T_true.graph["cell_assignments"], dtype=int)

In [ ]:
mutant_copy_numbers = adata.uns['mutant_copy_numbers']

In [ ]:
mutant_copy_numbers_df = pd.DataFrame(mutant_copy_numbers, columns=adata.var_names, index=[f"Clone_{i}" for i in range(mutant_copy_numbers.shape[0])])

In [ ]:
plot_clone_fractions_bar(cell_samples, cell_assignments, save_path=os.path.join(fig_directory, dataset, "simulation_overview.svg"))

In [ ]:
op.pl.show_tree(T_true)

In [ ]:
mutant_copy_numbers_df

In [ ]:
variant = "SNV3"
sample = 2

In [ ]:
LoPhy_results = run_LoPhy_with_missing_variant(adata, 
                                         case_study_folder=dataset_path, 
                                         case_study_name=f"LoPhy_original_data")

In [ ]:
LoPhy_results_missing_var = run_LoPhy_with_missing_variant(adata, 
                                         case_study_folder=dataset_path, 
                                         case_study_name=f"LoPhy_{variant}_missing_sample{sample}",
                                         variant_to_drop=variant, 
                                         sample_to_drop_from=sample, 
                                         amplicon_dropout=False)

In [ ]:
print(f"Original data\n MCN-MAE: {LoPhy_results[2]} \nTCN-MAE: {LoPhy_results[3]}")
print(f"Missing variant data\n MCN-MAE: {LoPhy_results_missing_var[2]} \nTCN-MAE: {LoPhy_results_missing_var[3]}")

In [ ]:
T_original_data = LoPhy_results[0][op.ul.DATA.MUTATION_TREE].copy()
T_missing_variant = LoPhy_results_missing_var[0][op.ul.DATA.MUTATION_TREE].copy()

In [ ]:
_, T_original_data = op.io.load_dot(os.path.join(dataset_path, "LoPhy_original_data", "out_ml0.gv"), 
                                   _type="cell_tree")
_, T_missing_variant = op.io.load_dot(os.path.join(dataset_path, f"LoPhy_{variant}_missing_sample{sample}", "out_ml0.gv"), 
                                   _type="cell_tree")

In [ ]:
T_original_data = prep_tree(T_original_data)
T_missing_variant = prep_tree(T_missing_variant)

In [ ]:
T_true.nodes["0"]["label"] = "root"

In [ ]:
T_true_relabeled = relabel_nodes(T_true)

In [ ]:
for u, v in T_true_relabeled.edges():
    T_true_relabeled[u][v][0].pop("weight", None)
    T_true_relabeled[u][v][0].pop("penwidth", None)

In [ ]:
op.pl.show_tree(T_true_relabeled, save_path=os.path.join(fig_directory, dataset, "true_tree.svg"))

In [ ]:
op.pl.show_tree(T_original_data, save_path=os.path.join(fig_directory, dataset, "LoPhy_tree_original_data.svg"))

In [ ]:
op.pl.show_tree(T_missing_variant, save_path=os.path.join(fig_directory, dataset, "LoPhy_tree_missing_variant.svg"))